In [2]:
# Transaction History Analyzer
LAST_NAME = "Campos"
SEED = 7
FAVORITE_ARTIST = "Car Seat Headrest"

def artist_positions(artist):
    positions = []
    for character in artist.lower():
        if character.isalpha():
            positions.append(ord(character) - ord("a") + 1)
    return positions

ARTIST_POSITIONS = artist_positions(FAVORITE_ARTIST)

def generated_number(index, offset=0, minimum=1, maximum=100):
    letter_value = ARTIST_POSITIONS[index % len(ARTIST_POSITIONS)]
    span = maximum - minimum + 1
    return minimum + (
        (SEED + letter_value + index * 3 + offset) % span
    )

# Display data in a formatted table
def print_table(headers, rows):
    rows = [[str(value) for value in row] for row in rows]
    headers = [str(header) for header in headers]
    widths = [
        max(
            len(headers[i]),
            max((len(row[i]) for row in rows), default=0)
        )
        for i in range(len(headers))
    ]

    print(" | ".join(
        headers[i].ljust(widths[i])
        for i in range(len(headers))
    ))
    print("-+-".join("-" * width for width in widths))
    for row in rows:
        print(" | ".join(
            row[i].ljust(widths[i])
            for i in range(len(headers))
        ))

# 1. Generate transaction records
customers = [f"{LAST_NAME}_C{i}" for i in range(1, 6)]
prices = {
    "Notebook": 35.00,
    "Pen": 15.00,
    "USB": 250.00,
    "Headphones": 450.00,
    "Calculator": 180.00
}

# Each tuple contains customer index, product, and quantity index.
purchase_plan = [
    (0, "Notebook", 0), (0, "Pen", 1), (0, "USB", 2),
    (1, "Notebook", 3), (1, "Headphones", 4), (1, "Calculator", 5),
    (2, "Pen", 6), (2, "USB", 7), (2, "Headphones", 8),
    (3, "Notebook", 9), (3, "Calculator", 10), (3, "USB", 11),
    (4, "Pen", 12), (4, "Headphones", 13), (4, "Calculator", 14),
    (0, "Calculator", 15), (1, "Pen", 16), (2, "Notebook", 17),
    (3, "Headphones", 18), (4, "USB", 19)
]

records = []
for customer_index, product, quantity_index in purchase_plan:
    quantity = generated_number(
        quantity_index,
        offset=customer_index,
        minimum=1,
        maximum=5
    )
    price = prices[product]
    records.append({
        "customer": customers[customer_index],
        "product": product,
        "quantity": quantity,
        "unit_price": price,
        "transaction_value": quantity * price
    })

# Required repeated complete transaction
records.append(records[0].copy())
print("=== EXERCISE 3: TRANSACTION HISTORY ANALYZER ===")

# 1. Generated transaction records
print("\n1. GENERATED TRANSACTION RECORDS")

print_table(
    ["Customer", "Product", "Quantity", "Unit Price", "Transaction Value"],
    [
        [
            r["customer"], r["product"], r["quantity"],
            f"{r['unit_price']:.2f}",
            f"{r['transaction_value']:.2f}"
        ]
        for r in records
    ]
)

# 2. Organized transaction data
organized = sorted(
    records,
    key=lambda r: (r["customer"], r["product"])
)
print("\n2. ORGANIZED TRANSACTION DATA")
print_table(
    ["Customer", "Product", "Quantity", "Unit Price", "Transaction Value"],
    [
        [
            r["customer"], r["product"], r["quantity"],
            f"{r['unit_price']:.2f}",
            f"{r['transaction_value']:.2f}"
        ]
        for r in organized
    ]
)

# 3. Detect repeated customer-product pairs and exact records
pair_counts = {}
exact_counts = {}
for r in records:
    pair = (r["customer"], r["product"])
    pair_counts[pair] = pair_counts.get(pair, 0) + 1
    exact = (
        r["customer"], r["product"], r["quantity"],
        r["unit_price"], r["transaction_value"]
    )
    exact_counts[exact] = exact_counts.get(exact, 0) + 1
repeated_pairs = {
    pair: count for pair, count in pair_counts.items()
    if count > 1
}
repeated_exact = {
    transaction: count for transaction, count in exact_counts.items()
    if count > 1
}

print("\n3. REPEATED OR DISTINCT TRANSACTIONS")
print_table(
    ["Summary Item", "Result"],
    [
        ["Total transaction records", len(records)],
        ["Distinct customers", len({r["customer"] for r in records})],
        ["Distinct products", len({r["product"] for r in records})],
        ["Distinct customer-product combinations", len(pair_counts)],
        [
            "Repeated customer-product combinations",
            len(repeated_pairs)
        ],
        [
            "Exact repeated transaction records",
            sum(count - 1 for count in repeated_exact.values())
        ]
    ]
)

print("\nRepeated Customer-Product Combinations")
print_table(
    ["Customer", "Product", "Number of Purchases"],
    [
        [customer, product, count]
        for (customer, product), count in repeated_pairs.items()
    ]
)

# 4. Transaction values
print("\n4. TRANSACTION VALUES")
print_table(
    ["Customer", "Product", "Quantity", "Unit Price", "Total"],
    [
        [
            r["customer"], r["product"], r["quantity"],
            f"{r['unit_price']:.2f}",
            f"{r['transaction_value']:.2f}"
        ]
        for r in records
    ]
)

overall_total = sum(r["transaction_value"] for r in records)
print(f"\nOverall Transaction Value: {overall_total:.2f}")

# 5. Customer summaries
print("\n5. CUSTOMER SUMMARIES")
customer_summary = []
for customer in customers:
    customer_records = [
        r for r in records if r["customer"] == customer
    ]

    total = sum(r["transaction_value"] for r in customer_records)
    quantity_total = sum(r["quantity"] for r in customer_records)

    customer_summary.append([
        customer,
        len(customer_records),
        quantity_total,
        f"{total:.2f}"
    ])

print_table(
    ["Customer", "Transactions", "Total Items", "Total Spending"],
    customer_summary
)

# 6. Product summaries
print("\n6. PRODUCT SUMMARIES")
product_summary = []
for product, price in prices.items():
    product_records = [
        r for r in records if r["product"] == product
    ]

    quantity_total = sum(r["quantity"] for r in product_records)
    sales_total = sum(r["transaction_value"] for r in product_records)

    product_summary.append([
        product,
        len(product_records),
        quantity_total,
        f"{price:.2f}",
        f"{sales_total:.2f}"
    ])

print_table(
    ["Product", "Transactions", "Quantity Sold", "Unit Price", "Total Sales"],
    product_summary
)

# 7. Ordered results: highest transaction value first
print("\n7. ORDERED RESULTS (HIGHEST VALUE FIRST)")
ordered_results = sorted(
    records,
    key=lambda r: (
        -r["transaction_value"],
        r["customer"],
        r["product"]
    )
)

print_table(
    ["Transaction Value", "Customer", "Product", "Quantity"],
    [
        [
            f"{r['transaction_value']:.2f}",
            r["customer"],
            r["product"],
            r["quantity"]
        ]
        for r in ordered_results
    ]
)

# 8. Final consolidated transaction report
print("\n8. FINAL CONSOLIDATED TRANSACTION REPORT")
print_table(
    ["Summary Item", "Result"],
    [
        ["Customers", len(customers)],
        ["Products", len(prices)],
        ["Transaction records", len(records)],
        ["Distinct customer-product combinations", len(pair_counts)],
        ["Repeated customer-product combinations", len(repeated_pairs)],
        [
            "Exact repeated transaction records",
            sum(count - 1 for count in repeated_exact.values())
        ],
        ["Overall transaction value", f"{overall_total:.2f}"]
    ]
)

print("\nGENERATION INPUTS")
print(f"Last name: {LAST_NAME}")
print(f"Seed: {SEED}")
print(f"Favorite artist: {FAVORITE_ARTIST}")

=== EXERCISE 3: TRANSACTION HISTORY ANALYZER ===

1. GENERATED TRANSACTION RECORDS
Customer  | Product    | Quantity | Unit Price | Transaction Value
----------+------------+----------+------------+------------------
Campos_C1 | Notebook   | 1        | 35.00      | 35.00            
Campos_C1 | Pen        | 2        | 15.00      | 30.00            
Campos_C1 | USB        | 2        | 250.00     | 500.00           
Campos_C2 | Notebook   | 2        | 35.00      | 70.00            
Campos_C2 | Headphones | 1        | 450.00     | 450.00           
Campos_C2 | Calculator | 5        | 180.00     | 900.00           
Campos_C3 | Pen        | 3        | 15.00      | 45.00            
Campos_C3 | USB        | 4        | 250.00     | 1000.00          
Campos_C3 | Headphones | 4        | 450.00     | 1800.00          
Campos_C4 | Notebook   | 4        | 35.00      | 140.00           
Campos_C4 | Calculator | 5        | 180.00     | 900.00           
Campos_C4 | USB        | 2        | 250.00    